# Case Study 04 — Text Sentiment Classification

**Problem type:** NLP · **Dataset:** Synthetic sentiment reviews

Classical NLP pipeline (TF-IDF + Logistic Regression) as a foundation. For BERT fine-tuning, see the production extension.

In [ ]:
import sys
sys.path.append('../..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from src.evaluation import evaluate_classification, compare_models
from src.visualization import plot_confusion_matrix
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Step 1: Problem Framing

Classify text reviews as positive or negative sentiment. Use case: customer feedback analysis, product reviews, social media monitoring.

## Step 2: Generate Demo Data

For reproducibility, we use a synthetic dataset. In production, replace with IMDB, Yelp, or domain-specific data.

In [ ]:
np.random.seed(42)
pos_words = ['great','amazing','wonderful','loved','fantastic','excellent','perfect','best','brilliant','outstanding','awesome','superb']
neg_words = ['terrible','awful','worst','disappointing','bad','horrible','boring','waste','poor','weak','frustrating','annoying']
fillers = ['movie','film','story','acting','plot','character']

reviews, labels = [], []
for _ in range(1000):
    n_words = np.random.randint(8, 25)
    text = ' '.join(np.random.choice(pos_words, n_words//2).tolist() + np.random.choice(fillers, n_words//2).tolist())
    reviews.append(text); labels.append(1)
for _ in range(1000):
    n_words = np.random.randint(8, 25)
    text = ' '.join(np.random.choice(neg_words, n_words//2).tolist() + np.random.choice(fillers, n_words//2).tolist())
    reviews.append(text); labels.append(0)

data = pd.DataFrame({'text': reviews, 'label': labels})
print(f'Dataset: {len(data)} samples ({data.label.sum()} positive)')
data.head()

## Step 3: Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['text'], data['label'], test_size=0.2, random_state=42, stratify=data['label']
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Step 4-5: Model Comparison

TF-IDF features + classical classifiers.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

pipelines = {
    'TF-IDF + LogReg': Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'TF-IDF + Naive Bayes': Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', MultinomialNB())]),
    'TF-IDF + Linear SVM': Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), ('clf', LinearSVC(random_state=42))]),
}

results = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append(evaluate_classification(y_test, preds, name=name))

compare_models(results).round(4)

## Step 6: Best Model Top Features

Logistic regression coefficients reveal the most predictive words.

In [ ]:
best = pipelines['TF-IDF + LogReg']
feature_names = best.named_steps['tfidf'].get_feature_names_out()
coefs = best.named_steps['clf'].coef_[0]

top_pos_idx = np.argsort(coefs)[-15:]
top_neg_idx = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh([feature_names[i] for i in top_pos_idx], coefs[top_pos_idx], color='green')
axes[0].set_title('Top Positive Words', fontweight='bold')
axes[1].barh([feature_names[i] for i in top_neg_idx], coefs[top_neg_idx], color='red')
axes[1].set_title('Top Negative Words', fontweight='bold')
plt.tight_layout()
plt.savefig('results/04_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7-8: Production Extension — BERT

For higher accuracy on real-world data, fine-tune BERT:

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
# ... tokenize, train, evaluate
```

**Typical results:** Classical (TF-IDF) achieves ~85% F1 on IMDB. BERT fine-tuning pushes to ~93%.

**Reflection:**
- TF-IDF is fast, interpretable, and surprisingly strong for sentiment
- BERT is heavier but captures context and negation better
- For production, monitor for distribution shift (slang evolves)

---
[← Back to main README](../../README.md)